# BC5CDR Corpus Keyword Extraction with KeyBERT

Description: Extract BC5CDR Corpus abstract keywords using KeyBERT and calculate associated importance scores.

In [1]:
# Imports
import json
from keybert import KeyBERT
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
import pandas as pd

In [2]:
# Path to GENIA corpus abstracts
infile_path = '../1-preprocessing/bc5cdr-preprocessed-no-lex.json'

# Load the JSON contents into a Python variable
with open(infile_path, 'r') as file:
    docs = json.load(file)

n_docs = len(docs)
print(n_docs)

1500


In [3]:
# Extract keywords with KeyBERT

# This vectorizer ensures that KeyBERT recognizes hyphens
vectorizer = CountVectorizer(token_pattern=r'(?u)\b[\w-]+\b')
kw_model = KeyBERT()
keywords_corpus = [] # Store a list of KeyBERT keywords, one list for each doc

# Extract keywords from each doc
for index, doc in enumerate(docs):
    if index < 3:
        print("------------------------------------------------------------------------")
        kw_out = kw_model.extract_keywords(doc, keyphrase_ngram_range=(1, 1), stop_words='english', top_n=25, highlight=True, vectorizer=vectorizer)
    else:
        kw_out = kw_model.extract_keywords(doc, keyphrase_ngram_range=(1, 1), stop_words='english', top_n=25, vectorizer=vectorizer)
    
    keywords_doc = []
   
    for kw in kw_out:
        keywords_doc.append(kw[0])
    
    keywords_corpus.append(keywords_doc) 

------------------------------------------------------------------------


naloxone reverses the antihypertensive effect of clonidine in unanesthetized spontaneously hypertensive rats the 
decrease in blood pressure and heart rate produced by intravenous clonidine five to twenty microgramskg was 
inhibited or reversed by nalozone two to two mgkg the hypotensive effect of one hundred mgkg alphamethyldopa was 
also partially reversed by naloxone naloxone alone did not affect either blood pressure or heart rate in brain 
membranes from spontaneously hypertensive rats clonidine one hundred and eight to one hundred and five m did not 
influence stereoselective binding of 3hnaloxone eight nm and naloxone one hundred and eight to one hundred and four
m did not influence clonidine suppressible binding of 3hdihydroergocryptine one nm these findings indicate that in 
spontaneously hypertensive rats the effects of central alphaadrenoceptor stimulation involve activation of opiate 
receptors as naloxone and clonidine do not appear to interact with the same receptor site the observed functional 
antagonism suggests the release of an endogenous opiate by clonidine or alphamethyldopa and the possible role of 
the opiate in the central control of sympathetic tone

------------------------------------------------------------------------


lidocaine induced cardiac_asystole intravenous administration of a single 50mg bolus of lidocaine in a 67yearold 
man resulted in profound depression of the activity of the sinoatrial and atrioventricular nodal pacemakers the 
patient had no apparent associated conditions which might have predisposed him to the development of 
bradyarrhythmias and thus this probably represented a true idiosyncrasy to lidocaine

------------------------------------------------------------------------


suxamethonium infusion rate and observed fasciculations a doseresponse study suxamethonium_chloride sch was 
administered iv to thirty-six adult males at six rates twenty-five mg s1 to twenty mg s1 the infusion was 
discontinued either when there was no muscular response to tetanic stimulation of the ulnar nerve or when sch one 
hundred and twenty mg was exceeded six additional patients received a 30mg iv bolus dose fasciculations in six 
areas of the body were scored from zero to three and summated as a total fasciculation score the times to first 
fasciculation twitch suppression and tetanus suppression were inversely related to the infusion rates 
fasciculations in the six areas and the total fasciculation score were related directly to the rate of infusion 
total fasciculation scores in the 30mg bolus group and the 5mg s1 and 20mg s1 infusion groups were not 
significantly different

In [4]:
# Calculate KeyBERT keyword scores

# Flatten, extract, and alphebetize unique keywords
unique_raw_keywords = list({keyword for doc in keywords_corpus for keyword in doc})
unique_raw_keywords.sort()
n_unique_raw_keywords = len(unique_raw_keywords)

# Create master data frame
master_df = pd.DataFrame(data={
    'raw_keyword': unique_raw_keywords,
    'KeyBERT': np.zeros(n_unique_raw_keywords)
})

# Calculate keyword scores
for idx, raw_keyword in master_df['raw_keyword'].items():
    score = sum(1 for doc in keywords_corpus if raw_keyword in doc) / n_docs
    master_df.at[idx, 'KeyBERT'] = score


display(master_df)

,raw_keyword,KeyBERT
0,0014unit,0.000667
1,00diisopropyl_phosphorofluoridate,0.000667
2,059mg,0.000667
3,05mgkg,0.000667
4,100mgkg,0.000667
...,...,...
10741,zopiclone,0.000667
10742,zuclopenthixol,0.000667
10743,zungconde,0.000667
10744,zyban,0.000667


In [5]:
# Recover BC5CDR lexical units from raw keywords

# Read in BC5CDR data
bc5cdr_keyword_path = '../1-preprocessing/bc5cdr-keywords.tsv'
bc5cdr_df = pd.read_csv(bc5cdr_keyword_path, sep="\t")

# Convert lexical units to lowercase and strip '_lex' postfix
bc5cdr_df['raw_keyword'] = bc5cdr_df['lex'].str.lower().str.removesuffix('_lex')

# Add scores for lexical units
result_df = pd.merge(master_df, bc5cdr_df, on='raw_keyword', how='outer')

# Replace NaN scores with 0
result_df['KeyBERT'] = result_df['KeyBERT'].fillna(0)

# Clean up resulting data frame
result_df.rename(columns={'lex': 'term'}, inplace=True)

# Condition: If 'term' == NaN, take 'raw_keyword'; otherwise keep 'term'
# Includes non-lexical term scores.
result_df['term'] = np.where(result_df['term'].isna(), result_df['raw_keyword'], result_df['term'])

# Replace any NaNs in the 'sem' column with an empty string
# Uses empty string for the semantic class of non-lexical terms
result_df['sem'] = result_df['sem'].fillna(str())

# Drop 'raw_keywords' column
result_df.drop(columns=['raw_keyword'], inplace=True)

# Reorder columns
result_df = result_df.reindex(columns=['term', 'sem', 'KeyBERT'])

# Write to TSV
result_df.to_csv('keybert-scores.tsv', sep='\t', index=False)

display(result_df)

,term,sem,KeyBERT
0,0014unit,,0.000667
1,00diisopropyl_phosphorofluoridate_lex,Chemical,0.000667
2,059mg,,0.000667
3,05mgkg,,0.000667
4,100mgkg,,0.000667
...,...,...,...
11193,penk_lex,Chemical,0.000000
11194,fhf_lex,Disease,0.000000
11195,cld_lex,Disease,0.000000
11196,ramipril_lex,Chemical,0.000000
